---
title: "Final Project: Classification"
format: 
  html:
    embed-resources: true
execute:
  echo: true
code-fold: true
author: James Compagno
jupyter: python3
---

# **Classification Models**

## **A. Models That Are Classification Only**

### **Logistic Regression**

* Binary: direct
* Multiclass: uses **OvR (One-vs-Rest)** by default in `sklearn`
* Can be configured to use **multinomial (softmax)**, but class notes emphasize OvR

---

### **LDA — Linear Discriminant Analysis**

* Works for **binary and multiclass**
* **Native multiclass** (no OvR or OvO needed)
* Models each class using Normal distributions with shared variance

---

### **QDA — Quadratic Discriminant Analysis**

* Works for **binary and multiclass**
* **Native multiclass** (no OvR or OvO)
* Similar to LDA but allows different variances per class

---

### **SVC — Support Vector Classifier (linear kernel)**

* Binary: standard SVC
* Multiclass (in `sklearn`): **OvR by default**

```python
SVC(decision_function_shape='ovr')
```

---

### **SVM — Kernel Support Vector Machine (RBF, polynomial, etc.)**

* Same base estimator as SVC in `sklearn`
* Binary: standard SVM
* Multiclass: **OvR by default**
* OvO is possible in theory or other implementations

# **GSB 544 – Fall 2025 – Classification**

### *Predict a person's political affiliation from their responses to survey questions.*

---

## **The Data**

This dataset comes from a survey conducted in **March 2018** by Survey Sampling International, commissioned by **Cards Against Humanity**.

The survey includes demographic questions and opinion-based questions relating to legality, morality, sexuality, religion, and political attitudes.

The **bolded question is the target variable** for prediction.

---

## **Survey Questions Included in the Dataset (with column names)**

* **Q1** — *What gender do you identify with?*
* **Q2** — *Age*
* **political_affiliation** — **In politics today, do you consider yourself a Democrat, a Republican, or Independent?** *(Target variable)*
* **Q4** — *Would you say you are liberal, conservative, or moderate?*
* **Q5** — *What is your highest level of education?*
* **Q6** — *What is your race?*
* **Q7** — *Do you believe that prostitution should be against the law?*
* **Q8** — *Do you believe that it should be against the law to smoke marijuana for recreation?*
* **Q9** — *Do you believe the sale of human organs for transplants should be against the law?*
* **Q10** — *Would you consider yourself a religious person?*
* **Q11** — *Would you consider yourself pro-life or pro-choice?*
* **Q12** — *Do you generally disapprove of people who pursue casual sex, hookups, or one-night stands?*
* **Q13** — *Based on your general impressions, would you say that people who smoke marijuana tend to have more casual sex than people who abstain?*
* **Q14** — *If abortion became strictly banned, do you think women would become more willing to have casual sex, less willing, or behave no differently?*
* **Q15** — *Do you agree/disagree: “A woman should have the right to do what she wants with her own body.”*
* **Q16** — *Do you agree/disagree: “Abortion is morally wrong.”*
* **Q17** — *Do you agree/disagree: “Sex without love is okay.”*
* **Q18** — *Do you believe an elected official who has committed sexual misconduct in their personal life can still behave ethically in office?*

In [179]:
import numpy as np
import pandas as pd
import plotnine as p9
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import (
    LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
)
from sklearn.svm import SVC
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score,
    precision_recall_fscore_support, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc, precision_score,
    recall_score, f1_score
)
from sklearn.model_selection import (
    GridSearchCV, cross_val_score, train_test_split, cross_val_predict
)
from sklearn.pipeline import Pipeline

In [180]:
# Read the data
CAH_train = pd.read_csv("/Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-classification/CAH-201803-train.csv")

# Load data and prepare features/target
X = CAH_train.drop(["id_num", "political_affiliation"], axis=1)
y = CAH_train["political_affiliation"]

# Model Library 
model_library = {}
records = []

# Read test data
CAH_test = pd.read_csv("/Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-classification/CAH-201803-test.csv")
X_test = CAH_test.drop("id_num", axis=1)
test_ids = CAH_test["id_num"]

In [181]:
# Define feature lists
numerical_features = ["Q2", "Q15", "Q16", "Q17"] 

categorical_features = ["Q1", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9", 
                        "Q10", "Q11", "Q12", "Q13", "Q14", "Q18"]

# Column Transformer 
ct = ColumnTransformer(
    [
        ("standardize", 
         StandardScaler(), 
         numerical_features),
        ("encode",
         OneHotEncoder(drop='first', sparse_output=False),
         categorical_features)
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# Q1: KNN

In [182]:
def knn_gridsearch(model_name, features=None, k_range=None, weights=None, p_values=None):
    """
    Uses GridSearchCV to find the best hyperparameters for KNN (multi-class)
    model_name - will be stored in model_library
    k_range - list of k values to test 
    weights - list of weight options ['uniform', 'distance']
    p_values - list of p values for Minkowski distance [1, 2]
    features - list or None 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Set defaults if not provided
    if k_range is None:
        k_range = [5]
    if weights is None:
        weights = ['uniform']
    if p_values is None:
        p_values = [2]
    
    # Pipeline with KNN Classifier
    pipe = Pipeline([
        ("preprocess", ct),
        ("knn", KNeighborsClassifier())
    ])
    
    # Parameter grid - now includes multiple hyperparameters
    param_grid = {
        'knn__n_neighbors': k_range,
        'knn__weights': weights,
        'knn__p': p_values
    }
    
    # GridSearchCV - using ACCURACY as primary metric
    grid_search = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring='accuracy',
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_subset, y)
    
    # Store best model
    best_model = grid_search.best_estimator_
    model_library[model_name] = best_model
    
    # Best parameters
    best_k = grid_search.best_params_['knn__n_neighbors']
    best_weights = grid_search.best_params_['knn__weights']
    best_p = grid_search.best_params_['knn__p']
    
    # Cross-validated predictions
    y_pred_cv = cross_val_predict(best_model, X_subset, y, cv=5)
    y_proba_cv = cross_val_predict(best_model, X_subset, y, cv=5, method='predict_proba')
    
    # Metrics on CV predictions (multi-class)
    conf_matrix = confusion_matrix(y, y_pred_cv)
    cv_accuracy = accuracy_score(y, y_pred_cv)
    
    # Multi-class ROC AUC (One-vs-Rest)
    cv_roc_auc = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
    
    # Precision, Recall, F1 (weighted average across classes)
    precision = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
    recall = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
    
    # Store results
    records.append({
        "Model": model_name,
        "Classification Type": "KNN",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Hyperparameter 1 Name": "n_neighbors", 
        "Hyperparameter 1 Value": best_k,
        "Hyperparameter 2 Name": "weights", 
        "Hyperparameter 2 Value": best_weights,
        "Hyperparameter 3 Name": "p",
        "Hyperparameter 3 Value": best_p,
        "Range Tested (k)": str(k_range),
        "Range Tested (weights)": str(weights),
        "Range Tested (p)": str(p_values),
        "CV Accuracy": cv_accuracy,
        "ROC AUC (weighted)": cv_roc_auc,
        "Precision (weighted)": precision,
        "Recall (weighted)": recall,
        "F1 (weighted)": f1,
        "Confusion Matrix": conf_matrix,
    })
    
    # Print results
    print(f"Best Parameters:")
    print(f"  n_neighbors (K): {best_k}")
    print(f"  weights: {best_weights}")
    print(f"  p (distance metric): {best_p} ({'manhattan' if best_p == 1 else 'euclidean' if best_p == 2 else 'minkowski'})")
    print(f"\nCV Accuracy: {cv_accuracy:.4f}")
    print(f"ROC AUC (weighted): {cv_roc_auc:.4f}")
    print(f"\nConfusion Matrix (CV):")
    print(conf_matrix)
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred_cv))

    return

In [183]:
knn_gridsearch(
    "KNN_broad", 
    features=None, 
    k_range=list(range(1, 51, 2)),  
    weights=['uniform', 'distance'],
    p_values=[1, 2]
)

Best Parameters:
  n_neighbors (K): 13
  weights: distance
  p (distance metric): 1 (manhattan)

CV Accuracy: 0.5740
ROC AUC (weighted): 0.7487

Confusion Matrix (CV):
[[34 16  9]
 [18 30  8]
 [ 8 13 33]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.57      0.58      0.57        59
 Independent       0.51      0.54      0.52        56
  Republican       0.66      0.61      0.63        54

    accuracy                           0.57       169
   macro avg       0.58      0.57      0.58       169
weighted avg       0.58      0.57      0.58       169



In [184]:
knn_gridsearch(
    "KNN_fine", 
    features=None, 
    k_range=list(range(7, 21)),  # 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20
    weights=['uniform', 'distance'],
    p_values=[1]
)

Best Parameters:
  n_neighbors (K): 16
  weights: distance
  p (distance metric): 1 (manhattan)

CV Accuracy: 0.5799
ROC AUC (weighted): 0.7503

Confusion Matrix (CV):
[[34 17  8]
 [18 30  8]
 [ 6 14 34]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.59      0.58      0.58        59
 Independent       0.49      0.54      0.51        56
  Republican       0.68      0.63      0.65        54

    accuracy                           0.58       169
   macro avg       0.59      0.58      0.58       169
weighted avg       0.58      0.58      0.58       169



In [185]:
dfKNN = pd.DataFrame(records)
dfKNN[dfKNN['Classification Type'] == 'KNN'].sort_values('CV Accuracy', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix
1,KNN_fine,KNN,All,n_neighbors,16,weights,distance,p,1,"[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, ...","['uniform', 'distance']",[1],0.579882,0.750268,0.584895,0.579882,0.581753,"[[34, 17, 8], [18, 30, 8], [6, 14, 34]]"
0,KNN_broad,KNN,All,n_neighbors,13,weights,distance,p,1,"[1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25...","['uniform', 'distance']","[1, 2]",0.573964,0.748652,0.577207,0.573964,0.575153,"[[34, 16, 9], [18, 30, 8], [8, 13, 33]]"


# Logistic Regression

In [186]:
def logistic_gridsearch(model_name, features=None, C_range=None, penalties=None, class_weights=None):
    """
    Uses GridSearchCV to find the best hyperparameters for Logistic Regression
    model_name - will be stored in model_library
    C_range - list of regularization strength values to test 
    penalties - list of penalty types ['l1', 'l2']
    class_weights - list of class weight options [None, 'balanced']
    features - list or None 
    """

    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Set defaults if not provided
    if C_range is None:
        C_range = [1.0]
    if penalties is None:
        penalties = ['l2']
    if class_weights is None:
        class_weights = [None]
    
    # Pipeline with Logistic Regression
    pipe = Pipeline([
        ("preprocess", ct),
        ("logistic", LogisticRegression(
            solver='saga',  # Supports both L1 and L2
            max_iter=2000,
            random_state=67
        ))
    ])
    
    # Parameter grid
    param_grid = {
        'logistic__C': C_range,
        'logistic__penalty': penalties,
        'logistic__class_weight': class_weights
    }
    
    # GridSearchCV - using ACCURACY as primary metric
    grid_search = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring='accuracy',
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_subset, y)
    
    # Store best model
    best_model = grid_search.best_estimator_
    model_library[model_name] = best_model
    
    # Best parameters
    best_C = grid_search.best_params_['logistic__C']
    best_penalty = grid_search.best_params_['logistic__penalty']
    best_class_weight = grid_search.best_params_['logistic__class_weight']
    
    # Cross-validated predictions
    y_pred_cv = cross_val_predict(best_model, X_subset, y, cv=5)
    y_proba_cv = cross_val_predict(best_model, X_subset, y, cv=5, method='predict_proba')
    
    # Metrics on CV predictions (multi-class)
    conf_matrix = confusion_matrix(y, y_pred_cv)
    cv_accuracy = accuracy_score(y, y_pred_cv)
    
    # Multi-class ROC AUC (One-vs-Rest)
    cv_roc_auc = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
    
    # Precision, Recall, F1 (weighted average across classes)
    precision = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
    recall = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
    
    # Store results
    records.append({
        "Model": model_name,
        "Classification Type": "Logistic Regression",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Hyperparameter 1 Name": "C", 
        "Hyperparameter 1 Value": best_C,
        "Hyperparameter 2 Name": "penalty",
        "Hyperparameter 2 Value": best_penalty,
        "Hyperparameter 3 Name": "class_weight",
        "Hyperparameter 3 Value": best_class_weight,
        "Range Tested (C)": str(C_range),
        "Range Tested (penalty)": str(penalties),
        "Range Tested (class_weight)": str(class_weights),
        "CV Accuracy": cv_accuracy,
        "ROC AUC (weighted)": cv_roc_auc,
        "Precision (weighted)": precision,
        "Recall (weighted)": recall,
        "F1 (weighted)": f1,
        "Confusion Matrix": conf_matrix,
    })
    
    # Print results
    print(f"Best Parameters:")
    print(f"  C: {best_C}")
    print(f"  penalty: {best_penalty}")
    print(f"  class_weight: {best_class_weight}")
    print(f"\nCV Accuracy: {cv_accuracy:.4f}")
    print(f"ROC AUC (weighted): {cv_roc_auc:.4f}")
    print(f"\nConfusion Matrix (CV):")
    print(conf_matrix)
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred_cv))

    return

In [187]:
logistic_gridsearch(
    "Logistic_v1", 
    features=None, 
    C_range=[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 100],  
    penalties=['l1', 'l2'],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 0.5
  penalty: l2
  class_weight: balanced

CV Accuracy: 0.6213
ROC AUC (weighted): 0.7747

Confusion Matrix (CV):
[[39 15  5]
 [16 30 10]
 [ 6 12 36]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.64      0.66      0.65        59
 Independent       0.53      0.54      0.53        56
  Republican       0.71      0.67      0.69        54

    accuracy                           0.62       169
   macro avg       0.62      0.62      0.62       169
weighted avg       0.62      0.62      0.62       169



/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max

In [188]:
dflog = pd.DataFrame(records)
dflog[dflog['Classification Type'] == 'Logistic Regression'].sort_values('CV Accuracy', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix,Range Tested (C),Range Tested (penalty),Range Tested (class_weight)
2,Logistic_v1,Logistic Regression,All,C,0.5,penalty,l2,class_weight,balanced,NaN,NaN,NaN,0.621302,0.774703,0.623152,0.621302,0.621971,"[[39, 15, 5], [16, 30, 10], [6, 12, 36]]","[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10,...","['l1', 'l2']","[None, 'balanced']"


# Decision Tree

In [189]:
def tree_gridsearch(model_name, features=None, max_depth_range=None, min_samples_leaf=None, min_samples_split=None):
    """
    Uses GridSearchCV to find the best hyperparameters for Decision Tree
    model_name - will be stored in model_library
    max_depth_range - list of max_depth values to test [None, 3, 5, 7]
    min_samples_leaf - list of min_samples_leaf values [1, 2, 4]
    min_samples_split - list of min_samples_split values [2, 5, 10]
    features - list or None 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Set defaults if not provided
    if max_depth_range is None:
        max_depth_range = [None]
    if min_samples_leaf is None:
        min_samples_leaf = [1]
    if min_samples_split is None:
        min_samples_split = [2]
    
    # Pipeline with Decision Tree
    pipe = Pipeline([
        ("preprocess", ct),
        ("tree", DecisionTreeClassifier(random_state=67))
    ])
    
    # Parameter grid
    param_grid = {
        'tree__max_depth': max_depth_range,
        'tree__min_samples_leaf': min_samples_leaf,
        'tree__min_samples_split': min_samples_split
    }
    
    # GridSearchCV - using ACCURACY as primary metric
    grid_search = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring='accuracy',
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_subset, y)
    
    # Store best model
    best_model = grid_search.best_estimator_
    model_library[model_name] = best_model
    
    # Best parameters
    best_max_depth = grid_search.best_params_['tree__max_depth']
    best_min_samples_leaf = grid_search.best_params_['tree__min_samples_leaf']
    best_min_samples_split = grid_search.best_params_['tree__min_samples_split']
    
    # Cross-validated predictions
    y_pred_cv = cross_val_predict(best_model, X_subset, y, cv=5)
    y_proba_cv = cross_val_predict(best_model, X_subset, y, cv=5, method='predict_proba')
    
    # Metrics on CV predictions (multi-class)
    conf_matrix = confusion_matrix(y, y_pred_cv)
    cv_accuracy = accuracy_score(y, y_pred_cv)
    
    # Multi-class ROC AUC (One-vs-Rest)
    cv_roc_auc = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
    
    # Precision, Recall, F1 (weighted average across classes)
    precision = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
    recall = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
    
    # Store results
    records.append({
        "Model": model_name,
        "Classification Type": "Decision Tree",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Hyperparameter 1 Name": "max_depth", 
        "Hyperparameter 1 Value": best_max_depth,
        "Hyperparameter 2 Name": "min_samples_leaf",
        "Hyperparameter 2 Value": best_min_samples_leaf,
        "Hyperparameter 3 Name": "min_samples_split",
        "Hyperparameter 3 Value": best_min_samples_split,
        "Range Tested (max_depth)": str(max_depth_range),
        "Range Tested (min_samples_leaf)": str(min_samples_leaf),
        "Range Tested (min_samples_split)": str(min_samples_split),
        "CV Accuracy": cv_accuracy,
        "ROC AUC (weighted)": cv_roc_auc,
        "Precision (weighted)": precision,
        "Recall (weighted)": recall,
        "F1 (weighted)": f1,
        "Confusion Matrix": conf_matrix,
    })
    
    # Print results
    print(f"Best Parameters:")
    print(f"  max_depth: {best_max_depth}")
    print(f"  min_samples_leaf: {best_min_samples_leaf}")
    print(f"  min_samples_split: {best_min_samples_split}")
    print(f"\nCV Accuracy: {cv_accuracy:.4f}")
    print(f"ROC AUC (weighted): {cv_roc_auc:.4f}")
    print(f"\nConfusion Matrix (CV):")
    print(conf_matrix)
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred_cv))

    return

In [190]:
tree_gridsearch(
    "Tree_v1", 
    features=None, 
    max_depth_range=[None, 3, 5, 7, 10],
    min_samples_leaf=[1, 2, 4],
    min_samples_split=[2, 5, 10]
)

Best Parameters:
  max_depth: None
  min_samples_leaf: 4
  min_samples_split: 10

CV Accuracy: 0.5266
ROC AUC (weighted): 0.6621

Confusion Matrix (CV):
[[33 16 10]
 [17 28 11]
 [12 14 28]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.53      0.56      0.55        59
 Independent       0.48      0.50      0.49        56
  Republican       0.57      0.52      0.54        54

    accuracy                           0.53       169
   macro avg       0.53      0.53      0.53       169
weighted avg       0.53      0.53      0.53       169



In [191]:
dflog = pd.DataFrame(records)
dflog[dflog['Classification Type'] == 'Decision Tree'].sort_values('CV Accuracy', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix,Range Tested (C),Range Tested (penalty),Range Tested (class_weight),Range Tested (max_depth),Range Tested (min_samples_leaf),Range Tested (min_samples_split)
3,Tree_v1,Decision Tree,All,max_depth,NaN,min_samples_leaf,4,min_samples_split,10,NaN,NaN,NaN,0.526627,0.662083,0.528372,0.526627,0.526922,"[[33, 16, 10], [17, 28, 11], [12, 14, 28]]",NaN,NaN,NaN,"[None, 3, 5, 7, 10]","[1, 2, 4]","[2, 5, 10]"


# LDA (Linear Discriminant Analysis)

In [192]:
def lda_gridsearch(model_name, features=None, solver=None, shrinkage=None):
    """
    Uses GridSearchCV to find the best hyperparameters for LDA
    model_name - will be stored in model_library
    solver - list of solver options ['svd', 'lsqr', 'eigen']
    shrinkage - list of shrinkage options [None, 'auto'] (only works with 'lsqr' or 'eigen')
    features - list or None 
    
    Note: LDA has limited hyperparameters. Shrinkage can help with high-dimensional data.
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Set defaults if not provided
    if solver is None:
        solver = ['svd']
    if shrinkage is None:
        shrinkage = [None]
    
    # Pipeline with LDA
    pipe = Pipeline([
        ("preprocess", ct),
        ("lda", LinearDiscriminantAnalysis())
    ])
    
    # Parameter grid
    # Note: shrinkage only works with 'lsqr' or 'eigen' solvers, not 'svd'
    param_grid = {
        'lda__solver': solver,
        'lda__shrinkage': shrinkage
    }
    
    # GridSearchCV - using ACCURACY as primary metric
    grid_search = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring='accuracy',
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_subset, y)
    
    # Store best model
    best_model = grid_search.best_estimator_
    model_library[model_name] = best_model
    
    # Best parameters
    best_solver = grid_search.best_params_['lda__solver']
    best_shrinkage = grid_search.best_params_['lda__shrinkage']
    
    # Cross-validated predictions
    y_pred_cv = cross_val_predict(best_model, X_subset, y, cv=5)
    y_proba_cv = cross_val_predict(best_model, X_subset, y, cv=5, method='predict_proba')
    
    # Metrics on CV predictions (multi-class)
    conf_matrix = confusion_matrix(y, y_pred_cv)
    cv_accuracy = accuracy_score(y, y_pred_cv)
    
    # Multi-class ROC AUC (One-vs-Rest)
    cv_roc_auc = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
    
    # Precision, Recall, F1 (weighted average across classes)
    precision = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
    recall = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
    
    # Store results
    records.append({
        "Model": model_name,
        "Classification Type": "LDA",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Hyperparameter 1 Name": "solver", 
        "Hyperparameter 1 Value": best_solver,
        "Hyperparameter 2 Name": "shrinkage",
        "Hyperparameter 2 Value": best_shrinkage,
        "Range Tested (solver)": str(solver),
        "Range Tested (shrinkage)": str(shrinkage),
        "CV Accuracy": cv_accuracy,
        "ROC AUC (weighted)": cv_roc_auc,
        "Precision (weighted)": precision,
        "Recall (weighted)": recall,
        "F1 (weighted)": f1,
        "Confusion Matrix": conf_matrix,
    })
    
    # Print results
    print(f"Best Parameters:")
    print(f"  solver: {best_solver}")
    print(f"  shrinkage: {best_shrinkage}")
    print(f"\nCV Accuracy: {cv_accuracy:.4f}")
    print(f"ROC AUC (weighted): {cv_roc_auc:.4f}")
    print(f"\nConfusion Matrix (CV):")
    print(conf_matrix)
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred_cv))

    return

In [193]:
lda_gridsearch(
    "LDA_v2", 
    features=None, 
    solver=['svd', 'lsqr', 'eigen'],
    shrinkage=[None, 'auto']  # 'auto' uses Ledoit-Wolf lemma
)

Best Parameters:
  solver: lsqr
  shrinkage: auto

CV Accuracy: 0.6095
ROC AUC (weighted): 0.7696

Confusion Matrix (CV):
[[39 14  6]
 [19 26 11]
 [ 5 11 38]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.62      0.66      0.64        59
 Independent       0.51      0.46      0.49        56
  Republican       0.69      0.70      0.70        54

    accuracy                           0.61       169
   macro avg       0.61      0.61      0.61       169
weighted avg       0.61      0.61      0.61       169



/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
5 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.f

In [194]:
dfLDA = pd.DataFrame(records)
dfLDA[dfLDA['Classification Type'] == 'LDA'].sort_values('CV Accuracy', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix,Range Tested (C),Range Tested (penalty),Range Tested (class_weight),Range Tested (max_depth),Range Tested (min_samples_leaf),Range Tested (min_samples_split),Range Tested (solver),Range Tested (shrinkage)
4,LDA_v2,LDA,All,solver,lsqr,shrinkage,auto,NaN,NaN,NaN,NaN,NaN,0.609467,0.769557,0.60581,0.609467,0.607027,"[[39, 14, 6], [19, 26, 11], [5, 11, 38]]",NaN,NaN,NaN,NaN,NaN,NaN,"['svd', 'lsqr', 'eigen']","[None, 'auto']"


# QDA (Quadratic Discriminant Analysis)

In [195]:
def qda_gridsearch(model_name, features=None, reg_param=None):
    """
    Uses GridSearchCV to find the best hyperparameters for QDA
    model_name - will be stored in model_library
    reg_param - list of regularization parameter values [0.0 to 1.0]
    features - list or None 
    
    Note: reg_param regularizes the covariance estimate, helping with singular covariance matrices.
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Set defaults if not provided
    if reg_param is None:
        reg_param = [0.0]
    
    # Pipeline with QDA
    pipe = Pipeline([
        ("preprocess", ct),
        ("qda", QuadraticDiscriminantAnalysis())
    ])
    
    # Parameter grid
    param_grid = {
        'qda__reg_param': reg_param
    }
    
    # GridSearchCV - using ACCURACY as primary metric
    grid_search = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring='accuracy',
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_subset, y)
    
    # Store best model
    best_model = grid_search.best_estimator_
    model_library[model_name] = best_model
    
    # Best parameters
    best_reg_param = grid_search.best_params_['qda__reg_param']
    
    # Cross-validated predictions
    y_pred_cv = cross_val_predict(best_model, X_subset, y, cv=5)
    y_proba_cv = cross_val_predict(best_model, X_subset, y, cv=5, method='predict_proba')
    
    # Metrics on CV predictions (multi-class)
    conf_matrix = confusion_matrix(y, y_pred_cv)
    cv_accuracy = accuracy_score(y, y_pred_cv)
    
    # Multi-class ROC AUC (One-vs-Rest)
    cv_roc_auc = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
    
    # Precision, Recall, F1 (weighted average across classes)
    precision = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
    recall = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
    
    # Store results
    records.append({
        "Model": model_name,
        "Classification Type": "QDA",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Hyperparameter 1 Name": "reg_param", 
        "Hyperparameter 1 Value": best_reg_param,
        "Range Tested (reg_param)": str(reg_param),
        "CV Accuracy": cv_accuracy,
        "ROC AUC (weighted)": cv_roc_auc,
        "Precision (weighted)": precision,
        "Recall (weighted)": recall,
        "F1 (weighted)": f1,
        "Confusion Matrix": conf_matrix,
    })
    
    # Print results
    print(f"Best Parameters:")
    print(f"  reg_param: {best_reg_param}")
    print(f"\nCV Accuracy: {cv_accuracy:.4f}")
    print(f"ROC AUC (weighted): {cv_roc_auc:.4f}")
    print(f"\nConfusion Matrix (CV):")
    print(conf_matrix)
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred_cv))

    return

In [196]:
qda_gridsearch(
    "QDA_v1", 
    features=None, 
    reg_param=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
)

Best Parameters:
  reg_param: 0.1

CV Accuracy: 0.5917
ROC AUC (weighted): 0.7520

Confusion Matrix (CV):
[[37 15  7]
 [21 27  8]
 [ 6 12 36]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.58      0.63      0.60        59
 Independent       0.50      0.48      0.49        56
  Republican       0.71      0.67      0.69        54

    accuracy                           0.59       169
   macro avg       0.59      0.59      0.59       169
weighted avg       0.59      0.59      0.59       169



/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


In [197]:
qda_gridsearch(
    "QDA_v2_fine", 
    features=None, 
    reg_param=[0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


Best Parameters:
  reg_param: 0.1

CV Accuracy: 0.5917
ROC AUC (weighted): 0.7520

Confusion Matrix (CV):
[[37 15  7]
 [21 27  8]
 [ 6 12 36]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.58      0.63      0.60        59
 Independent       0.50      0.48      0.49        56
  Republican       0.71      0.67      0.69        54

    accuracy                           0.59       169
   macro avg       0.59      0.59      0.59       169
weighted avg       0.59      0.59      0.59       169



In [198]:
qda_gridsearch(
    "QDA_v3_fine", 
    features=None, 
    reg_param=[0.0, 0.02, 0.04, 0.06, 0.08, 0.1, 0.12, 0.14, 0.16, 0.18, 0.2]
)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


Best Parameters:
  reg_param: 0.1

CV Accuracy: 0.5917
ROC AUC (weighted): 0.7520

Confusion Matrix (CV):
[[37 15  7]
 [21 27  8]
 [ 6 12 36]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.58      0.63      0.60        59
 Independent       0.50      0.48      0.49        56
  Republican       0.71      0.67      0.69        54

    accuracy                           0.59       169
   macro avg       0.59      0.59      0.59       169
weighted avg       0.59      0.59      0.59       169



In [199]:
dfQDA = pd.DataFrame(records)
dfQDA[dfQDA['Classification Type'] == 'QDA'].sort_values('CV Accuracy', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix,Range Tested (C),Range Tested (penalty),Range Tested (class_weight),Range Tested (max_depth),Range Tested (min_samples_leaf),Range Tested (min_samples_split),Range Tested (solver),Range Tested (shrinkage),Range Tested (reg_param)
5,QDA_v1,QDA,All,reg_param,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.591716,0.752011,0.593059,0.591716,0.591807,"[[37, 15, 7], [21, 27, 8], [6, 12, 36]]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, ..."
6,QDA_v2_fine,QDA,All,reg_param,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.591716,0.752011,0.593059,0.591716,0.591807,"[[37, 15, 7], [21, 27, 8], [6, 12, 36]]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0..."
7,QDA_v3_fine,QDA,All,reg_param,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.591716,0.752011,0.593059,0.591716,0.591807,"[[37, 15, 7], [21, 27, 8], [6, 12, 36]]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 0.02, 0.04, 0.06, 0.08, 0.1, 0.12, 0.14,..."


# SVC (Linear SVM)

In [200]:
def svc_gridsearch(model_name, features=None, C_range=None, class_weights=None):
    """
    Uses GridSearchCV to find the best hyperparameters for Linear SVC
    model_name - will be stored in model_library
    C_range - list of regularization strength values [0.01, 0.1, 1, 10]
    class_weights - list of class weight options [None, 'balanced']
    features - list or None 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Set defaults if not provided
    if C_range is None:
        C_range = [1.0]
    if class_weights is None:
        class_weights = [None]
    
    # Pipeline with Linear SVC
    pipe = Pipeline([
        ("preprocess", ct),
        ("svc", SVC(
            kernel='linear', 
            probability=True,  # Needed for predict_proba
            random_state=67
        ))
    ])
    
    # Parameter grid
    param_grid = {
        'svc__C': C_range,
        'svc__class_weight': class_weights
    }
    
    # GridSearchCV - using ACCURACY as primary metric
    grid_search = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring='accuracy',
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_subset, y)
    
    # Store best model
    best_model = grid_search.best_estimator_
    model_library[model_name] = best_model
    
    # Best parameters
    best_C = grid_search.best_params_['svc__C']
    best_class_weight = grid_search.best_params_['svc__class_weight']
    
    # Cross-validated predictions
    y_pred_cv = cross_val_predict(best_model, X_subset, y, cv=5)
    y_proba_cv = cross_val_predict(best_model, X_subset, y, cv=5, method='predict_proba')
    
    # Metrics on CV predictions (multi-class)
    conf_matrix = confusion_matrix(y, y_pred_cv)
    cv_accuracy = accuracy_score(y, y_pred_cv)
    
    # Multi-class ROC AUC (One-vs-Rest)
    cv_roc_auc = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
    
    # Precision, Recall, F1 (weighted average across classes)
    precision = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
    recall = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
    
    # Store results
    records.append({
        "Model": model_name,
        "Classification Type": "SVC (Linear)",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Hyperparameter 1 Name": "C", 
        "Hyperparameter 1 Value": best_C,
        "Hyperparameter 2 Name": "class_weight",
        "Hyperparameter 2 Value": best_class_weight,
        "Range Tested (C)": str(C_range),
        "Range Tested (class_weight)": str(class_weights),
        "CV Accuracy": cv_accuracy,
        "ROC AUC (weighted)": cv_roc_auc,
        "Precision (weighted)": precision,
        "Recall (weighted)": recall,
        "F1 (weighted)": f1,
        "Confusion Matrix": conf_matrix,
    })
    
    # Print results
    print(f"Best Parameters:")
    print(f"  C: {best_C}")
    print(f"  class_weight: {best_class_weight}")
    print(f"\nCV Accuracy: {cv_accuracy:.4f}")
    print(f"ROC AUC (weighted): {cv_roc_auc:.4f}")
    print(f"\nConfusion Matrix (CV):")
    print(conf_matrix)
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred_cv))

    return

In [201]:
svc_gridsearch(
    "SVC_v1", 
    features=None, 
    C_range=[0.01, 0.1, 1, 10],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 10
  class_weight: None

CV Accuracy: 0.5917
ROC AUC (weighted): 0.7421

Confusion Matrix (CV):
[[39 14  6]
 [23 25  8]
 [ 7 11 36]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.57      0.66      0.61        59
 Independent       0.50      0.45      0.47        56
  Republican       0.72      0.67      0.69        54

    accuracy                           0.59       169
   macro avg       0.60      0.59      0.59       169
weighted avg       0.59      0.59      0.59       169



In [202]:
svc_gridsearch(
    "SVC_5s", 
    features=None, 
    C_range=[5, 10, 15, 20, 25, 30, 35],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 10
  class_weight: None

CV Accuracy: 0.5917
ROC AUC (weighted): 0.7421

Confusion Matrix (CV):
[[39 14  6]
 [23 25  8]
 [ 7 11 36]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.57      0.66      0.61        59
 Independent       0.50      0.45      0.47        56
  Republican       0.72      0.67      0.69        54

    accuracy                           0.59       169
   macro avg       0.60      0.59      0.59       169
weighted avg       0.59      0.59      0.59       169



In [203]:
svc_gridsearch(
    "SVC_v2_fine", 
    features=None, 
    C_range=[1, 2, 5, 7, 10, 15, 20, 25, 30, 50],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 2
  class_weight: balanced

CV Accuracy: 0.6154
ROC AUC (weighted): 0.7712

Confusion Matrix (CV):
[[41 16  2]
 [20 28  8]
 [ 8 11 35]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.59      0.69      0.64        59
 Independent       0.51      0.50      0.50        56
  Republican       0.78      0.65      0.71        54

    accuracy                           0.62       169
   macro avg       0.63      0.61      0.62       169
weighted avg       0.62      0.62      0.62       169



In [204]:
svc_gridsearch(
    "SVC_v3_fine", 
    features=None, 
    C_range=[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 7
  class_weight: None

CV Accuracy: 0.6036
ROC AUC (weighted): 0.7486

Confusion Matrix (CV):
[[40 15  4]
 [21 27  8]
 [ 8 11 35]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.58      0.68      0.62        59
 Independent       0.51      0.48      0.50        56
  Republican       0.74      0.65      0.69        54

    accuracy                           0.60       169
   macro avg       0.61      0.60      0.60       169
weighted avg       0.61      0.60      0.60       169



In [205]:
svc_gridsearch(
    "SVC_v5_fine", 
    features=None, 
    C_range=[0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 2
  class_weight: balanced

CV Accuracy: 0.6154
ROC AUC (weighted): 0.7712

Confusion Matrix (CV):
[[41 16  2]
 [20 28  8]
 [ 8 11 35]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.59      0.69      0.64        59
 Independent       0.51      0.50      0.50        56
  Republican       0.78      0.65      0.71        54

    accuracy                           0.62       169
   macro avg       0.63      0.61      0.62       169
weighted avg       0.62      0.62      0.62       169



In [206]:
svc_gridsearch(
    "SVC_v6_fine", 
    features=None, 
    C_range=[1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 2.0
  class_weight: balanced

CV Accuracy: 0.6154
ROC AUC (weighted): 0.7712

Confusion Matrix (CV):
[[41 16  2]
 [20 28  8]
 [ 8 11 35]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.59      0.69      0.64        59
 Independent       0.51      0.50      0.50        56
  Republican       0.78      0.65      0.71        54

    accuracy                           0.62       169
   macro avg       0.63      0.61      0.62       169
weighted avg       0.62      0.62      0.62       169



In [207]:
dflog = pd.DataFrame(records)
dflog[dflog['Classification Type'] == 'SVC (Linear)'].sort_values('CV Accuracy', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix,Range Tested (C),Range Tested (penalty),Range Tested (class_weight),Range Tested (max_depth),Range Tested (min_samples_leaf),Range Tested (min_samples_split),Range Tested (solver),Range Tested (shrinkage),Range Tested (reg_param)
10,SVC_v2_fine,SVC (Linear),All,C,2,class_weight,balanced,NaN,NaN,NaN,NaN,NaN,0.615385,0.771189,0.624657,0.615385,0.616751,"[[41, 16, 2], [20, 28, 8], [8, 11, 35]]","[1, 2, 5, 7, 10, 15, 20, 25, 30, 50]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN
12,SVC_v5_fine,SVC (Linear),All,C,2,class_weight,balanced,NaN,NaN,NaN,NaN,NaN,0.615385,0.771189,0.624657,0.615385,0.616751,"[[41, 16, 2], [20, 28, 8], [8, 11, 35]]","[0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN
13,SVC_v6_fine,SVC (Linear),All,C,2.0,class_weight,balanced,NaN,NaN,NaN,NaN,NaN,0.615385,0.771189,0.624657,0.615385,0.616751,"[[41, 16, 2], [20, 28, 8], [8, 11, 35]]","[1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, ...",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN
11,SVC_v3_fine,SVC (Linear),All,C,7,class_weight,None,NaN,NaN,NaN,NaN,NaN,0.603550,0.748578,0.609136,0.603550,0.603810,"[[40, 15, 4], [21, 27, 8], [8, 11, 35]]","[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN
8,SVC_v1,SVC (Linear),All,C,10,class_weight,None,NaN,NaN,NaN,NaN,NaN,0.591716,0.742080,0.593064,0.591716,0.590253,"[[39, 14, 6], [23, 25, 8], [7, 11, 36]]","[0.01, 0.1, 1, 10]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN
9,SVC_5s,SVC (Linear),All,C,10,class_weight,None,NaN,NaN,NaN,NaN,NaN,0.591716,0.742080,0.593064,0.591716,0.590253,"[[39, 14, 6], [23, 25, 8], [7, 11, 36]]","[5, 10, 15, 20, 25, 30, 35]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN


# SVM

In [208]:
def svm_gridsearch(model_name, features=None, C_range=None, gamma_range=None, class_weights=None):
    """
    Uses GridSearchCV to find the best hyperparameters for SVM
    model_name - will be stored in model_library
    C_range - list of regularization strength values [0.1, 1, 5, 10]
    gamma_range - list of gamma values ['scale', 'auto', 0.01, 0.1]
    class_weights - list of class weight options [None, 'balanced']
    features - list or None 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Set defaults if not provided
    if C_range is None:
        C_range = [1.0]
    if gamma_range is None:
        gamma_range = ['scale']
    if class_weights is None:
        class_weights = [None]
    
    # Pipeline with SVM
    pipe = Pipeline([
        ("preprocess", ct),
        ("svm", SVC(
            kernel='rbf', 
            probability=True,  # Needed for predict_proba
            random_state=67
        ))
    ])
    
    # Parameter grid
    param_grid = {
        'svm__C': C_range,
        'svm__gamma': gamma_range,
        'svm__class_weight': class_weights
    }
    
    # GridSearchCV - using ACCURACY as primary metric
    grid_search = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring='accuracy',
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_subset, y)
    
    # Store best model
    best_model = grid_search.best_estimator_
    model_library[model_name] = best_model
    
    # Best parameters
    best_C = grid_search.best_params_['svm__C']
    best_gamma = grid_search.best_params_['svm__gamma']
    best_class_weight = grid_search.best_params_['svm__class_weight']
    
    # Cross-validated predictions
    y_pred_cv = cross_val_predict(best_model, X_subset, y, cv=5)
    y_proba_cv = cross_val_predict(best_model, X_subset, y, cv=5, method='predict_proba')
    
    # Metrics on CV predictions (multi-class)
    conf_matrix = confusion_matrix(y, y_pred_cv)
    cv_accuracy = accuracy_score(y, y_pred_cv)
    
    # Multi-class ROC AUC (One-vs-Rest)
    cv_roc_auc = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
    
    # Precision, Recall, F1 (weighted average across classes)
    precision = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
    recall = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
    
    # Store results
    records.append({
        "Model": model_name,
        "Classification Type": "SVM",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Hyperparameter 1 Name": "C", 
        "Hyperparameter 1 Value": best_C,
        "Hyperparameter 2 Name": "gamma",
        "Hyperparameter 2 Value": best_gamma,
        "Hyperparameter 3 Name": "class_weight",
        "Hyperparameter 3 Value": best_class_weight,
        "Range Tested (C)": str(C_range),
        "Range Tested (gamma)": str(gamma_range),
        "Range Tested (class_weight)": str(class_weights),
        "CV Accuracy": cv_accuracy,
        "ROC AUC (weighted)": cv_roc_auc,
        "Precision (weighted)": precision,
        "Recall (weighted)": recall,
        "F1 (weighted)": f1,
        "Confusion Matrix": conf_matrix,
    })
    
    # Print results
    print(f"Best Parameters:")
    print(f"  C: {best_C}")
    print(f"  gamma: {best_gamma}")
    print(f"  class_weight: {best_class_weight}")
    print(f"\nCV Accuracy: {cv_accuracy:.4f}")
    print(f"ROC AUC (weighted): {cv_roc_auc:.4f}")
    print(f"\nConfusion Matrix (CV):")
    print(conf_matrix)
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred_cv))

    return

In [209]:
svm_gridsearch(
    "SVM_v1", 
    features=None, 
    C_range=[0.1, 1, 5, 10, 15, 20],
    gamma_range=['scale', 'auto', 0.01, 0.1],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 5
  gamma: auto
  class_weight: balanced

CV Accuracy: 0.6036
ROC AUC (weighted): 0.7685

Confusion Matrix (CV):
[[42 14  3]
 [17 28 11]
 [ 9 13 32]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.62      0.71      0.66        59
 Independent       0.51      0.50      0.50        56
  Republican       0.70      0.59      0.64        54

    accuracy                           0.60       169
   macro avg       0.61      0.60      0.60       169
weighted avg       0.61      0.60      0.60       169



In [210]:
svm_gridsearch(
    "SVM_v2", 
    features=None, 
    C_range=[1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0],
    gamma_range=['scale', 'auto', 0.01, 0.1],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 2.8
  gamma: 0.1
  class_weight: None

CV Accuracy: 0.5858
ROC AUC (weighted): 0.7683

Confusion Matrix (CV):
[[39 16  4]
 [21 25 10]
 [ 7 12 35]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.58      0.66      0.62        59
 Independent       0.47      0.45      0.46        56
  Republican       0.71      0.65      0.68        54

    accuracy                           0.59       169
   macro avg       0.59      0.59      0.59       169
weighted avg       0.59      0.59      0.59       169



In [211]:
svm_gridsearch(
    "SVM_v3_fine", 
    features=None, 
    C_range=[3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5],
    gamma_range=['scale', 'auto', 0.01, 0.1],
    class_weights=[None, 'balanced']
)

Best Parameters:
  C: 5.0
  gamma: auto
  class_weight: balanced

CV Accuracy: 0.6036
ROC AUC (weighted): 0.7685

Confusion Matrix (CV):
[[42 14  3]
 [17 28 11]
 [ 9 13 32]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.62      0.71      0.66        59
 Independent       0.51      0.50      0.50        56
  Republican       0.70      0.59      0.64        54

    accuracy                           0.60       169
   macro avg       0.61      0.60      0.60       169
weighted avg       0.61      0.60      0.60       169



In [212]:
dfSVM = pd.DataFrame(records)
dfSVM[dfSVM['Classification Type'] == 'SVM'].sort_values('CV Accuracy', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix,Range Tested (C),Range Tested (penalty),Range Tested (class_weight),Range Tested (max_depth),Range Tested (min_samples_leaf),Range Tested (min_samples_split),Range Tested (solver),Range Tested (shrinkage),Range Tested (reg_param),Range Tested (gamma)
14,SVM_v1,SVM,All,C,5,gamma,auto,class_weight,balanced,NaN,NaN,NaN,0.603550,0.768513,0.606601,0.603550,0.602579,"[[42, 14, 3], [17, 28, 11], [9, 13, 32]]","[0.1, 1, 5, 10, 15, 20]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,"['scale', 'auto', 0.01, 0.1]"
16,SVM_v3_fine,SVM,All,C,5.0,gamma,auto,class_weight,balanced,NaN,NaN,NaN,0.603550,0.768513,0.606601,0.603550,0.602579,"[[42, 14, 3], [17, 28, 11], [9, 13, 32]]","[3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,"['scale', 'auto', 0.01, 0.1]"
15,SVM_v2,SVM,All,C,2.8,gamma,0.1,class_weight,None,NaN,NaN,NaN,0.585799,0.768262,0.587750,0.585799,0.585272,"[[39, 16, 4], [21, 25, 10], [7, 12, 35]]","[1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, ...",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,"['scale', 'auto', 0.01, 0.1]"


# All Models

In [213]:
dfAll = pd.DataFrame(records)
dfAll.sort_values('CV Accuracy', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix,Range Tested (C),Range Tested (penalty),Range Tested (class_weight),Range Tested (max_depth),Range Tested (min_samples_leaf),Range Tested (min_samples_split),Range Tested (solver),Range Tested (shrinkage),Range Tested (reg_param),Range Tested (gamma)
2,Logistic_v1,Logistic Regression,All,C,0.5,penalty,l2,class_weight,balanced,NaN,NaN,NaN,0.621302,0.774703,0.623152,0.621302,0.621971,"[[39, 15, 5], [16, 30, 10], [6, 12, 36]]","[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10,...","['l1', 'l2']","[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,SVC_v6_fine,SVC (Linear),All,C,2.0,class_weight,balanced,NaN,NaN,NaN,NaN,NaN,0.615385,0.771189,0.624657,0.615385,0.616751,"[[41, 16, 2], [20, 28, 8], [8, 11, 35]]","[1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, ...",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,SVC_v5_fine,SVC (Linear),All,C,2,class_weight,balanced,NaN,NaN,NaN,NaN,NaN,0.615385,0.771189,0.624657,0.615385,0.616751,"[[41, 16, 2], [20, 28, 8], [8, 11, 35]]","[0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,SVC_v2_fine,SVC (Linear),All,C,2,class_weight,balanced,NaN,NaN,NaN,NaN,NaN,0.615385,0.771189,0.624657,0.615385,0.616751,"[[41, 16, 2], [20, 28, 8], [8, 11, 35]]","[1, 2, 5, 7, 10, 15, 20, 25, 30, 50]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LDA_v2,LDA,All,solver,lsqr,shrinkage,auto,NaN,NaN,NaN,NaN,NaN,0.609467,0.769557,0.605810,0.609467,0.607027,"[[39, 14, 6], [19, 26, 11], [5, 11, 38]]",NaN,NaN,NaN,NaN,NaN,NaN,"['svd', 'lsqr', 'eigen']","[None, 'auto']",NaN,NaN
16,SVM_v3_fine,SVM,All,C,5.0,gamma,auto,class_weight,balanced,NaN,NaN,NaN,0.603550,0.768513,0.606601,0.603550,0.602579,"[[42, 14, 3], [17, 28, 11], [9, 13, 32]]","[3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,"['scale', 'auto', 0.01, 0.1]"
14,SVM_v1,SVM,All,C,5,gamma,auto,class_weight,balanced,NaN,NaN,NaN,0.603550,0.768513,0.606601,0.603550,0.602579,"[[42, 14, 3], [17, 28, 11], [9, 13, 32]]","[0.1, 1, 5, 10, 15, 20]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,"['scale', 'auto', 0.01, 0.1]"
11,SVC_v3_fine,SVC (Linear),All,C,7,class_weight,None,NaN,NaN,NaN,NaN,NaN,0.603550,0.748578,0.609136,0.603550,0.603810,"[[40, 15, 4], [21, 27, 8], [8, 11, 35]]","[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,SVC_5s,SVC (Linear),All,C,10,class_weight,None,NaN,NaN,NaN,NaN,NaN,0.591716,0.742080,0.593064,0.591716,0.590253,"[[39, 14, 6], [23, 25, 8], [7, 11, 36]]","[5, 10, 15, 20, 25, 30, 35]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,SVC_v1,SVC (Linear),All,C,10,class_weight,None,NaN,NaN,NaN,NaN,NaN,0.591716,0.742080,0.593064,0.591716,0.590253,"[[39, 14, 6], [23, 25, 8], [7, 11, 36]]","[0.01, 0.1, 1, 10]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [214]:
dfROCAUC = pd.DataFrame(records)
dfROCAUC.sort_values('ROC AUC (weighted)', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix,Range Tested (C),Range Tested (penalty),Range Tested (class_weight),Range Tested (max_depth),Range Tested (min_samples_leaf),Range Tested (min_samples_split),Range Tested (solver),Range Tested (shrinkage),Range Tested (reg_param),Range Tested (gamma)
2,Logistic_v1,Logistic Regression,All,C,0.5,penalty,l2,class_weight,balanced,NaN,NaN,NaN,0.621302,0.774703,0.623152,0.621302,0.621971,"[[39, 15, 5], [16, 30, 10], [6, 12, 36]]","[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10,...","['l1', 'l2']","[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,SVC_v5_fine,SVC (Linear),All,C,2,class_weight,balanced,NaN,NaN,NaN,NaN,NaN,0.615385,0.771189,0.624657,0.615385,0.616751,"[[41, 16, 2], [20, 28, 8], [8, 11, 35]]","[0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,SVC_v2_fine,SVC (Linear),All,C,2,class_weight,balanced,NaN,NaN,NaN,NaN,NaN,0.615385,0.771189,0.624657,0.615385,0.616751,"[[41, 16, 2], [20, 28, 8], [8, 11, 35]]","[1, 2, 5, 7, 10, 15, 20, 25, 30, 50]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,SVC_v6_fine,SVC (Linear),All,C,2.0,class_weight,balanced,NaN,NaN,NaN,NaN,NaN,0.615385,0.771189,0.624657,0.615385,0.616751,"[[41, 16, 2], [20, 28, 8], [8, 11, 35]]","[1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, ...",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LDA_v2,LDA,All,solver,lsqr,shrinkage,auto,NaN,NaN,NaN,NaN,NaN,0.609467,0.769557,0.605810,0.609467,0.607027,"[[39, 14, 6], [19, 26, 11], [5, 11, 38]]",NaN,NaN,NaN,NaN,NaN,NaN,"['svd', 'lsqr', 'eigen']","[None, 'auto']",NaN,NaN
16,SVM_v3_fine,SVM,All,C,5.0,gamma,auto,class_weight,balanced,NaN,NaN,NaN,0.603550,0.768513,0.606601,0.603550,0.602579,"[[42, 14, 3], [17, 28, 11], [9, 13, 32]]","[3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,"['scale', 'auto', 0.01, 0.1]"
14,SVM_v1,SVM,All,C,5,gamma,auto,class_weight,balanced,NaN,NaN,NaN,0.603550,0.768513,0.606601,0.603550,0.602579,"[[42, 14, 3], [17, 28, 11], [9, 13, 32]]","[0.1, 1, 5, 10, 15, 20]",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,"['scale', 'auto', 0.01, 0.1]"
15,SVM_v2,SVM,All,C,2.8,gamma,0.1,class_weight,None,NaN,NaN,NaN,0.585799,0.768262,0.587750,0.585799,0.585272,"[[39, 16, 4], [21, 25, 10], [7, 12, 35]]","[1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, ...",NaN,"[None, 'balanced']",NaN,NaN,NaN,NaN,NaN,NaN,"['scale', 'auto', 0.01, 0.1]"
5,QDA_v1,QDA,All,reg_param,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.591716,0.752011,0.593059,0.591716,0.591807,"[[37, 15, 7], [21, 27, 8], [6, 12, 36]]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, ...",NaN
6,QDA_v2_fine,QDA,All,reg_param,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.591716,0.752011,0.593059,0.591716,0.591807,"[[37, 15, 7], [21, 27, 8], [6, 12, 36]]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0...",NaN


# Final Model and Export

In [215]:
# Choose your best model
best_model_name = "Logistic_v1"  
final_model = model_library[best_model_name]

# Display the metrics for your chosen model
results_df = pd.DataFrame(records)
print("=== Model metrics ===")
print(results_df[results_df["Model"] == best_model_name][["Model", "CV Accuracy", "ROC AUC (weighted)", "F1 (weighted)"]].to_string(index=False))

=== Model metrics ===
      Model  CV Accuracy  ROC AUC (weighted)  F1 (weighted)
Logistic_v1     0.621302            0.774703       0.621971


In [216]:
# Make predictions on test data
y_test_pred = final_model.predict(X_test)

# Check prediction distribution
print("Prediction distribution:")
print(pd.Series(y_test_pred).value_counts())

Prediction distribution:
Independent    61
Democrat       59
Republican     46
Name: count, dtype: int64


In [217]:
# Build submission
submission = pd.DataFrame({
    "id_num": test_ids,
    "political_affiliation_predicted": y_test_pred
})

# Preview
print(submission.head(10))

# Save to your project folder
submission_path = "/Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-classification/classification_submission.csv"
submission.to_csv(submission_path, index=False)
print(f"\nSaved submission to {submission_path}")

   id_num political_affiliation_predicted
0       2                      Republican
1       3                        Democrat
2       4                        Democrat
3       6                      Republican
4      11                     Independent
5      12                        Democrat
6      13                      Republican
7      14                        Democrat
8      17                      Republican
9      21                        Democrat

Saved submission to /Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-classification/classification_submission.csv
